# 06. U10 Plate Dataset V2

B00 중심 원판 학습 데이터를 서로 다른 사람과 배경으로 확장하고, 같은 A/B/C 정답으로 U9와 U10을 비교합니다. 이 노트북은 **데이터 확장 → 학습 → 고정 평가 → 채택 판단**을 담당합니다.

In [ ]:
from hashlib import sha256
from pathlib import Path
import json
import math
import os
import resource
import shutil
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from snatch_analysis.config import B00_VIDEO_ID, DETECTION_MODEL_NAME
from snatch_analysis.metadata import (
    save_video_manifest,
    set_has_plate_labels,
)
from snatch_analysis.plate import (
    aggregate_plate_evaluations,
    cancel_plate_label,
    evaluate_plate_video,
    infer_plate_video_candidates,
    sample_time_bin_frames,
    save_plate_candidate_cache,
    save_plate_image,
    save_plate_track,
    save_u10_plate_dataset_yaml,
    save_yolo_plate_label,
    select_primary_plate_track,
    validate_pilot_ground_truth,
    validate_u10_plate_selection,
    yolo_bbox_to_opencv,
)
from snatch_analysis.pipeline import save_json_atomically
from snatch_analysis.pose import (
    resolve_inference_device,
    verify_ultralytics_safe_load,
)
from snatch_analysis.video import iter_video_frames, read_video_frame


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.")
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## 1. 설정

처음에는 모든 dictionary를 비우고 실행 flag를 `False`로 둡니다. 후보 contact sheet를 본 뒤 선택한 영상만 기록합니다. **54개 전체의 원판 유무를 먼저 라벨링하지 않습니다.**

In [ ]:
U10_HAS_PLATE_LABELS: dict[str, str] = {
    "a04_sn_f_01": "yes",
    "a05_sn_f_01": "yes",
    "b03_sn_f_01": "yes",
    "c02_sn_f_01": "yes",
    "a07_sn_f_01": "yes",
    "c08_sn_f_01": "yes",
}
U10_VIDEO_ROLES: dict[str, str] = {
    "a04_sn_f_01": "train",
    "a05_sn_f_01": "train",
    "b03_sn_f_01": "train",
    "c02_sn_f_01": "train",
    "a07_sn_f_01": "validation",
    "c08_sn_f_01": "validation",
}
U10_SELECTION_REASONS: dict[str, str] = {
    "a04_sn_f_01": "큰 초록 원판과 주변 기구가 함께 있는 복잡한 배경",
    "a05_sn_f_01": "카메라에 가까운 원판과 다른 촬영 거리",
    "b03_sn_f_01": "검은 원판과 밝은 벽 배경",
    "c02_sn_f_01": "다른 체형과 검은 원판 크기",
    "a07_sn_f_01": "검증용 초록 원판과 후방 사람·기구",
    "c08_sn_f_01": "검증용 새로운 사람과 카메라 거리",
}

SAVE_U10_METADATA = False
RUN_U10_EXTRACTION = False
RUN_U10_LABELING = False
OVERWRITE_U10_IMAGES = False
OVERWRITE_U10_LABELS = False

SAMPLES_PER_VIDEO = 20
MANIFEST_PATH = PROJECT_ROOT / "data/metadata/videos.csv"
BASELINE_DATASET_ROOT = PROJECT_ROOT / "data/plate_dataset"
DATASET_ROOT = PROJECT_ROOT / "data/plate_dataset_v2"
IMAGES_ROOT = DATASET_ROOT / "images"
LABELS_ROOT = DATASET_ROOT / "labels"
VIDEO_SPLIT_PATH = DATASET_ROOT / "video_split.csv"
SAMPLE_MANIFEST_PATH = DATASET_ROOT / "sample_frames.csv"
DATASET_YAML_PATH = DATASET_ROOT / "dataset.yaml"
U10_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts/u10_plate_dataset_v2"

if SAMPLES_PER_VIDEO != 20:
    raise ValueError("U10 비교에서는 영상당 정확히 20프레임을 사용합니다.")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest가 없습니다: {MANIFEST_PATH}")

## 2. 후보 영상 확인

평가 역할과 B00을 제외하고 `training_development` 사람마다 한 영상을 보여줍니다. 이 화면에서 원판 유무, 원판의 화면 내 크기, 배경과 카메라 거리를 확인합니다. A/B/C 균형보다 시각적 다양성이 우선입니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
eligible = manifest.loc[
    manifest["split_role"].eq("training_development")
    & manifest["read_status"].eq("ok")
    & manifest["lift"].eq("sn")
    & manifest["video_id"].ne(B00_VIDEO_ID)
    & manifest["has_plate"].ne("no")
].copy()
eligible["completion_order"] = eligible["completion"].map({"f": 0, "h": 1})
candidate_review = (
    eligible.sort_values(["person_id", "completion_order", "rep_index"])
    .groupby("person_id", as_index=False)
    .head(1)
    .sort_values(["group", "person_id"])
    .reset_index(drop=True)
)
if candidate_review.empty:
    raise ValueError("확인할 training_development 후보가 없습니다.")

column_count = 4
row_count = math.ceil(len(candidate_review) / column_count)
figure, axes = plt.subplots(row_count, column_count, figsize=(16, 4 * row_count))
axes = np.atleast_1d(axes).ravel()
for axis, row in zip(axes, candidate_review.itertuples(index=False)):
    frame_index = int(row.frame_count) // 2
    frame = read_video_frame(PROJECT_ROOT / row.path, frame_index)
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(
        f"{row.video_id} | person={row.person_id} | has_plate={row.has_plate}"
    )
    axis.axis("off")
for axis in axes[len(candidate_review):]:
    axis.axis("off")
plt.tight_layout()
plt.show()
display(
    candidate_review[[
        "video_id", "group", "person_id", "completion",
        "frame_count", "has_plate", "path",
    ]]
)

## 3. 선택 후보의 원판 유무 기록

contact sheet에서 직접 확인한 후보만 `U10_HAS_PLATE_LABELS`에 `yes` 또는 `no`로 입력합니다. 저장할 때만 `SAVE_U10_METADATA=True`로 바꾸고, 저장 후 다시 `False`로 되돌립니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
candidate_ids = set(candidate_review["video_id"].astype(str))
unknown_label_ids = sorted(set(U10_HAS_PLATE_LABELS) - candidate_ids)
if unknown_label_ids:
    raise ValueError("후보 contact sheet에 없는 video_id: " + ", ".join(unknown_label_ids))
if SAVE_U10_METADATA:
    if not U10_HAS_PLATE_LABELS:
        raise ValueError("저장할 U10_HAS_PLATE_LABELS가 비어 있습니다.")
    manifest = set_has_plate_labels(manifest, U10_HAS_PLATE_LABELS)
    save_video_manifest(manifest, MANIFEST_PATH)
    print(f"has_plate 저장 완료: {len(U10_HAS_PLATE_LABELS)}개 영상")
else:
    print("manifest 미저장: 확인 후 SAVE_U10_METADATA=True로 한 번 실행하세요.")
display(
    manifest.loc[
        manifest["video_id"].isin(U10_HAS_PLATE_LABELS),
        ["video_id", "person_id", "has_plate"],
    ]
)

## 4. 신규 6개 영상 역할 확정

`U10_VIDEO_ROLES`에 `train` 4개와 `validation` 2개를 입력합니다. 같은 사람은 두 역할로 나눌 수 없습니다. `U10_SELECTION_REASONS`에는 각 영상이 추가하는 원판 크기·색·배경·가림 차이를 한 줄로 기록합니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
if not U10_VIDEO_ROLES and not U10_SELECTION_REASONS:
    selected_videos = pd.DataFrame()
    print("U10 선택 대기: 후보 확인 후 두 dictionary에 6개 영상을 입력하세요.")
else:
    if set(U10_VIDEO_ROLES) != set(U10_SELECTION_REASONS):
        raise ValueError("U10_VIDEO_ROLES와 U10_SELECTION_REASONS의 video_id가 다릅니다.")
    manifest_by_video = manifest.set_index("video_id", drop=False)
    unknown_ids = sorted(set(U10_VIDEO_ROLES) - set(manifest_by_video.index.astype(str)))
    if unknown_ids:
        raise ValueError("Manifest에 없는 video_id: " + ", ".join(unknown_ids))
    selection = pd.DataFrame(
        [
            {
                "video_id": video_id,
                "person_id": str(manifest_by_video.loc[video_id, "person_id"]),
                "dataset_role": role,
                "selection_reason": U10_SELECTION_REASONS[video_id],
            }
            for video_id, role in U10_VIDEO_ROLES.items()
        ],
        columns=("video_id", "person_id", "dataset_role", "selection_reason"),
    )
    selection = validate_u10_plate_selection(
        selection, manifest, baseline_video_id=B00_VIDEO_ID
    )
    selected_videos = selection.merge(
        manifest[[
            "video_id", "group", "path", "fps", "frame_count",
            "width", "height", "completion", "has_plate",
        ]],
        on="video_id",
        how="left",
        validate="one_to_one",
    )
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    selected_videos.to_csv(VIDEO_SPLIT_PATH, index=False)
    display(selected_videos)

## 5. 시간 구간별 120프레임 표본

정확한 phase event를 새로 입력하지 않습니다. 각 영상을 시간상 다섯 구간으로 나누고 구간마다 4프레임을 선택합니다. 이 `sampling_bin`은 detector 데이터 다양성을 위한 구간이며 동작 phase 라벨이 아닙니다.

In [ ]:
if not VIDEO_SPLIT_PATH.exists():
    sample_frames = pd.DataFrame()
    print("sample 생성 대기: 먼저 6개 영상 selection을 저장하세요.")
else:
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_tables = [
        sample_time_bin_frames(
            row.video_id,
            int(row.frame_count),
            sample_count=SAMPLES_PER_VIDEO,
        ).assign(dataset_role=row.dataset_role)
        for row in selected_videos.itertuples(index=False)
    ]
    sample_frames = pd.concat(sample_tables, ignore_index=True)
    if len(sample_frames) != 120:
        raise RuntimeError(f"U10 sample은 120개여야 합니다: {len(sample_frames)}")
    if sample_frames.duplicated(["video_id", "frame_index"]).any():
        raise RuntimeError("U10 sample에 중복 video/frame이 있습니다.")
    sample_frames["image_filename"] = sample_frames.apply(
        lambda row: f"{row.video_id}_frame_{int(row.frame_index):04d}.jpg",
        axis=1,
    )
    sample_frames["label_filename"] = sample_frames["image_filename"].str.replace(
        ".jpg", ".txt", regex=False
    )
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    sample_frames.to_csv(SAMPLE_MANIFEST_PATH, index=False)
    display(
        sample_frames.groupby(["dataset_role", "video_id", "sampling_bin"])
        .size()
        .rename("sample_count")
        .reset_index()
    )

## 6. 이미지 추출

`RUN_U10_EXTRACTION=True`일 때만 파일을 만듭니다. 신규 영상은 한 번씩 순차 디코딩하고 선택 frame만 저장합니다. 기존 B00 30개 이미지·라벨은 원본을 건드리지 않고 V2 train에 복사합니다.

In [ ]:
extraction_actions = []
if not RUN_U10_EXTRACTION:
    print("이미지 미생성: sample 표를 확인한 뒤 RUN_U10_EXTRACTION=True로 바꾸세요.")
else:
    if not VIDEO_SPLIT_PATH.exists() or not SAMPLE_MANIFEST_PATH.exists():
        raise FileNotFoundError("video_split.csv와 sample_frames.csv가 필요합니다.")
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    for role in ("train", "validation"):
        (IMAGES_ROOT / role).mkdir(parents=True, exist_ok=True)
        (LABELS_ROOT / role).mkdir(parents=True, exist_ok=True)

    baseline_images = sorted((BASELINE_DATASET_ROOT / "images").glob("*.jpg"))
    baseline_labels = sorted((BASELINE_DATASET_ROOT / "labels").glob("*.txt"))
    if len(baseline_images) != 30 or len(baseline_labels) != 30:
        raise RuntimeError("B00 baseline 이미지와 라벨은 각각 30개여야 합니다.")
    for source_path in [*baseline_images, *baseline_labels]:
        destination_root = IMAGES_ROOT if source_path.suffix == ".jpg" else LABELS_ROOT
        destination_path = destination_root / "train" / source_path.name
        if destination_path.exists():
            action = "skipped"
        else:
            shutil.copy2(source_path, destination_path)
            action = "created"
        extraction_actions.append({"source": str(source_path), "action": action})

    metadata_by_video = selected_videos.set_index("video_id")
    for video_id, video_samples in sample_frames.groupby("video_id"):
        metadata = metadata_by_video.loc[video_id]
        targets = video_samples.set_index("frame_index")
        saved_indices = set()
        for frame_index, _, frame in iter_video_frames(
            PROJECT_ROOT / metadata.path,
            expected_frame_count=int(metadata.frame_count),
        ):
            if frame_index not in targets.index:
                continue
            sample = targets.loc[frame_index]
            image_path = IMAGES_ROOT / sample.dataset_role / sample.image_filename
            action = save_plate_image(
                image_path, frame, overwrite=OVERWRITE_U10_IMAGES
            )
            extraction_actions.append(
                {"video_id": video_id, "frame_index": frame_index, "action": action}
            )
            saved_indices.add(frame_index)
        missing_indices = sorted(set(targets.index.astype(int)) - saved_indices)
        if missing_indices:
            raise RuntimeError(f"{video_id} 추출 누락 frame: {missing_indices}")
    display(pd.DataFrame(extraction_actions)["action"].value_counts().rename("count"))

## 7. 신규 120프레임 수동 bbox

`RUN_U10_LABELING=True`로 바꾸면 아직 라벨이 없는 이미지부터 OpenCV ROI 창이 열립니다. 주동작자가 드는 카메라 쪽 원판 하나만 지정합니다. 취소한 frame은 좌표를 만들지 않고 `plate_label_cancelled`로 표에 남습니다. 재라벨링(`OVERWRITE_U10_LABELS=True`) 중 취소하면 잘못된 기존 bbox를 삭제합니다.

In [ ]:
label_actions = []
if not RUN_U10_LABELING:
    print("라벨링 미실행: 이미지 검토 후 RUN_U10_LABELING=True로 바꾸세요.")
else:
    if not SAMPLE_MANIFEST_PATH.exists():
        raise FileNotFoundError("sample_frames.csv가 필요합니다.")
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    try:
        for row in sample_frames.itertuples(index=False):
            image_path = IMAGES_ROOT / row.dataset_role / row.image_filename
            label_path = LABELS_ROOT / row.dataset_role / row.label_filename
            if label_path.exists() and not OVERWRITE_U10_LABELS:
                label_actions.append(
                    {"video_id": row.video_id, "frame_index": row.frame_index, "action": "skipped"}
                )
                continue
            frame = cv2.imread(str(image_path))
            if frame is None:
                raise FileNotFoundError(f"추출 이미지가 없습니다: {image_path}")
            bbox = cv2.selectROI(
                f"plate | {row.video_id} | frame={int(row.frame_index)}",
                frame,
                showCrosshair=True,
                fromCenter=False,
            )
            if bbox[2] <= 0 or bbox[3] <= 0:
                action = cancel_plate_label(
                    label_path, overwrite=OVERWRITE_U10_LABELS
                )
            else:
                action = save_yolo_plate_label(
                    label_path,
                    tuple(float(value) for value in bbox),
                    image_width=frame.shape[1],
                    image_height=frame.shape[0],
                    overwrite=OVERWRITE_U10_LABELS,
                )
            label_actions.append(
                {"video_id": row.video_id, "frame_index": row.frame_index, "action": action}
            )
    finally:
        cv2.destroyAllWindows()
    display(pd.DataFrame(label_actions))

## 8. bbox contact sheet 검수

신규 120개 라벨을 20개씩 나눠 contact sheet로 저장합니다. 초록 bbox가 주동작 원판 하나를 가리키는지 모두 확인합니다. B00 30개는 U6에서 이미 검수했으므로 이 화면에서는 신규 데이터만 다시 봅니다.

In [ ]:
contact_sheet_paths = []
if not SAMPLE_MANIFEST_PATH.exists():
    print("contact sheet 대기: sample_frames.csv가 없습니다.")
else:
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    if len(sample_frames) != 120:
        raise RuntimeError(f"신규 U10 sample은 120개여야 합니다: {len(sample_frames)}")
    review_entries = []
    missing_labels = []
    for row in sample_frames.itertuples(index=False):
        image_path = IMAGES_ROOT / row.dataset_role / row.image_filename
        label_path = LABELS_ROOT / row.dataset_role / row.label_filename
        if not image_path.exists() or not label_path.exists():
            missing_labels.append((row.video_id, int(row.frame_index)))
            continue
        frame = cv2.imread(str(image_path))
        if frame is None:
            raise OSError(f"이미지를 읽을 수 없습니다: {image_path}")
        tokens = label_path.read_text(encoding="utf-8").strip().split()
        if len(tokens) != 5 or tokens[0] != "0":
            raise ValueError(f"잘못된 YOLO 원판 라벨: {label_path}")
        bbox = yolo_bbox_to_opencv(
            tuple(float(value) for value in tokens[1:]),
            image_width=frame.shape[1],
            image_height=frame.shape[0],
        )
        review_entries.append((row, frame, bbox))
    print(f"신규 라벨: {len(review_entries)}/120, 미완료: {len(missing_labels)}")
    if missing_labels:
        display(pd.DataFrame(missing_labels, columns=["video_id", "frame_index"]))
    else:
        U10_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
        page_size = 20
        column_count = 5
        for page_index, start in enumerate(range(0, len(review_entries), page_size), start=1):
            page_entries = review_entries[start:start + page_size]
            row_count = math.ceil(len(page_entries) / column_count)
            figure, axes = plt.subplots(row_count, column_count, figsize=(16, 3.5 * row_count))
            axes = np.atleast_1d(axes).ravel()
            for axis, (row, frame, bbox) in zip(axes, page_entries):
                x, y, width, height = bbox
                marked = frame.copy()
                cv2.rectangle(marked, (x, y), (x + width, y + height), (0, 255, 0), 3)
                axis.imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB))
                axis.set_title(f"{row.video_id} | frame={int(row.frame_index)}")
                axis.axis("off")
            for axis in axes[len(page_entries):]:
                axis.axis("off")
            plt.tight_layout()
            output_path = U10_ARTIFACT_ROOT / f"label_contact_sheet_{page_index:02d}.png"
            figure.savefig(output_path, dpi=120, bbox_inches="tight")
            contact_sheet_paths.append(output_path)
            plt.show()
            plt.close(figure)
        display(pd.DataFrame({"contact_sheet_path": contact_sheet_paths}))

## 9. 학습용 dataset YAML

신규 120개와 B00 30개가 모두 준비됐을 때만 YAML을 씁니다. train은 B00 30개와 신규 train 80개, validation은 학습에 포함되지 않은 두 사람의 40개입니다. 현재 split의 파일명과 정확히 일치하지 않으면 이전 YAML도 무효화합니다.

In [ ]:
if not VIDEO_SPLIT_PATH.exists() or not SAMPLE_MANIFEST_PATH.exists():
    DATASET_YAML_PATH.unlink(missing_ok=True)
    print("dataset.yaml 대기: video_split.csv와 sample_frames.csv가 필요합니다.")
else:
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    dataset_counts = save_u10_plate_dataset_yaml(
        DATASET_ROOT,
        BASELINE_DATASET_ROOT,
        selected_videos,
        sample_frames,
    )
    print(f"dataset YAML 저장: {DATASET_YAML_PATH}")
    display(pd.DataFrame([dataset_counts]))

## 10. V2 원판 detector 학습 설정

M2 Air 8GB에서 먼저 비교할 수 있도록 YOLOv8n, 320px, batch 4, 10 epoch로 고정합니다. 기본값은 실행하지 않으며, 앞의 여섯 contact sheet에서 120개 bbox를 모두 확인한 경우에만 두 확인값을 `True`로 바꿉니다.

In [ ]:
RUN_U10_TRAINING = False
CONFIRM_U10_LABEL_REVIEW = False
U10_EPOCHS = 10
U10_IMAGE_SIZE = 320
U10_BATCH = 4
U10_SEED = 0
U10_REQUESTED_DEVICE = "mps"

U10_TRAINING_ROOT = U10_ARTIFACT_ROOT / "training"
U10_WEIGHTS_PATH = U10_TRAINING_ROOT / "weights/best.pt"
U10_RUN_SUMMARY_PATH = U10_TRAINING_ROOT / "run_summary.json"

## 11. 데이터 재검증 후 학습

학습 직전에 현재 split, 150개 image-label 쌍과 train/validation 사람 분리를 다시 검증합니다. 완료되면 `best.pt`와 재현에 필요한 설정·데이터 수·장치·학습 시간·weight SHA256을 `run_summary.json`에 저장합니다. `peak_rss_bytes`는 현재 macOS Python 프로세스의 누적 최대 메모리이므로 다른 OS 결과와 직접 비교하지 않습니다.

In [ ]:
u10_training_status = "not_run"
if not RUN_U10_TRAINING:
    print("U10 학습 대기: contact sheet 검수 후 두 확인값을 True로 바꾸세요.")
elif not CONFIRM_U10_LABEL_REVIEW:
    raise RuntimeError("contact sheet 6장을 모두 검수한 뒤 CONFIRM_U10_LABEL_REVIEW=True로 바꾸세요.")
else:
    U10_RUN_SUMMARY_PATH.unlink(missing_ok=True)
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    manifest = pd.read_csv(MANIFEST_PATH)
    validated_videos = validate_u10_plate_selection(
        selected_videos.loc[:, ["video_id", "person_id", "dataset_role", "selection_reason"]],
        manifest,
        baseline_video_id=B00_VIDEO_ID,
    )
    dataset_counts = save_u10_plate_dataset_yaml(
        DATASET_ROOT,
        BASELINE_DATASET_ROOT,
        selected_videos,
        sample_frames,
    )
    train_rows = validated_videos.loc[validated_videos["dataset_role"].eq("train")]
    validation_rows = validated_videos.loc[validated_videos["dataset_role"].eq("validation")]
    train_person_ids = sorted(train_rows["person_id"].astype(str).unique())
    validation_person_ids = sorted(validation_rows["person_id"].astype(str).unique())
    person_overlap = sorted(set(train_person_ids) & set(validation_person_ids))
    if person_overlap:
        raise ValueError(f"train/validation person_id 누수: {person_overlap}")

    device = resolve_inference_device(U10_REQUESTED_DEVICE)
    safe_load = verify_ultralytics_safe_load()
    import ultralytics

    previous_weights_mtime_ns = (
        U10_WEIGHTS_PATH.stat().st_mtime_ns if U10_WEIGHTS_PATH.exists() else None
    )
    training_started = time.perf_counter()
    training_model = ultralytics.YOLO(DETECTION_MODEL_NAME)
    training_model.train(
        data=str(DATASET_YAML_PATH),
        epochs=U10_EPOCHS,
        imgsz=U10_IMAGE_SIZE,
        batch=U10_BATCH,
        seed=U10_SEED,
        device=device["actual_device"],
        project=str(U10_TRAINING_ROOT.parent.resolve()),
        name=U10_TRAINING_ROOT.name,
        exist_ok=True,
    )
    training_seconds = time.perf_counter() - training_started
    produced_weights_path = Path(training_model.trainer.best)
    if produced_weights_path.resolve() != U10_WEIGHTS_PATH.resolve():
        raise RuntimeError(f"예상하지 않은 weight 경로: {produced_weights_path}")
    if not U10_WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"학습이 best.pt를 만들지 못했습니다: {U10_WEIGHTS_PATH}")
    if (
        previous_weights_mtime_ns is not None
        and U10_WEIGHTS_PATH.stat().st_mtime_ns == previous_weights_mtime_ns
    ):
        raise RuntimeError("현재 학습이 best.pt를 갱신하지 않았습니다.")

    run_summary = {
        "status": "complete",
        "dataset_yaml": str(DATASET_YAML_PATH.resolve()),
        "dataset_split": {
            "train_video_ids": sorted(train_rows["video_id"].astype(str)),
            "validation_video_ids": sorted(validation_rows["video_id"].astype(str)),
            "train_person_ids": train_person_ids,
            "validation_person_ids": validation_person_ids,
            "person_overlap": person_overlap,
        },
        "label_counts": dataset_counts,
        "model": DETECTION_MODEL_NAME,
        "ultralytics_version": ultralytics.__version__,
        "weights_path": str(U10_WEIGHTS_PATH.resolve()),
        "weights_sha256": sha256(U10_WEIGHTS_PATH.read_bytes()).hexdigest(),
        "model_size_bytes": U10_WEIGHTS_PATH.stat().st_size,
        "epochs": U10_EPOCHS,
        "batch": U10_BATCH,
        "image_size": U10_IMAGE_SIZE,
        "seed": U10_SEED,
        "requested_device": device["requested_device"],
        "actual_device": device["actual_device"],
        "device_fallback_reason": device["fallback_reason"],
        "training_seconds": training_seconds,
        "peak_rss_bytes": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss),
        "peak_rss_scope": "current macOS Python process accumulated peak",
        "safe_load": safe_load,
    }
    save_json_atomically(run_summary, U10_RUN_SUMMARY_PATH)
    u10_training_status = "complete"
    display(pd.DataFrame([{
        "status": u10_training_status,
        "weights": U10_WEIGHTS_PATH,
        "training_seconds": training_seconds,
        "actual_device": device["actual_device"],
        **dataset_counts,
    }]))

## 12. A/B/C 고정 평가 설정

U9에서 만든 A/B/C 30개 정답, confidence 0.10, image size 320과 기존 tracker를 그대로 사용합니다. 첫 실행은 후보만 저장하고, 초기 후보 그림을 확인한 뒤 세 영상의 U10_PRIMARY_PLATE_CHOICES를 입력합니다.

In [ ]:
RUN_U10_EVALUATION_INFERENCE = False
CONFIRM_U10_PRIMARY_PLATES = False
U10_EVALUATION_CONFIDENCE = 0.10
U10_EVALUATION_IMAGE_SIZE = 320
U10_PRIMARY_PLATE_CHOICES: dict[str, tuple[int, int] | None] = {}
PLATE_EVALUATION_IOU_THRESHOLD = 0.50
U10_FAILURE_REVIEW_LIMIT = 30

EVALUATION_SPLIT_PATH = PROJECT_ROOT / "data/splits/evaluation_videos.csv"
U10_GROUND_TRUTH_PATH = PROJECT_ROOT / "data/annotations/pilot_frame_ground_truth.csv"
U9_BASELINE_ROOT = PROJECT_ROOT / "artifacts/u9_baseline_conf010"
U9_COMPARISON_PATH = PROJECT_ROOT / "artifacts/u9_threshold_evaluation/plate_threshold_comparison.csv"
U10_EVALUATION_ROOT = U10_ARTIFACT_ROOT / "evaluation"
U10_FRAME_EVALUATION_PATH = U10_EVALUATION_ROOT / "plate_frame_evaluation.csv"
U10_MODEL_COMPARISON_PATH = U10_EVALUATION_ROOT / "plate_model_comparison.csv"
U10_DECISION_PATH = U10_EVALUATION_ROOT / "decision.json"

## 13. 후보 추론과 기존 tracker 평가

RUN_U10_EVALUATION_INFERENCE=True일 때만 U10 best.pt로 세 pilot 전체 후보를 다시 만듭니다. 후보 그림 확인 후에는 이 flag를 False로 되돌리고 CONFIRM_U10_PRIMARY_PLATES=True로 실행합니다. Pose는 U9에 저장된 주동작자 중심만 재사용합니다.

In [ ]:
evaluation_split = pd.read_csv(EVALUATION_SPLIT_PATH)
pilot_rows = (
    evaluation_split.loc[evaluation_split["split_role"].eq("model_selection_pilot")]
    .sort_values("group")
    .reset_index(drop=True)
)
if set(pilot_rows["read_status"]) != {"ok"}:
    raise ValueError("읽기 오류가 있는 pilot 영상은 평가할 수 없습니다.")
u10_ground_truth = validate_pilot_ground_truth(
    pd.read_csv(U10_GROUND_TRUTH_PATH), evaluation_split
)
training_summary = json.loads(U10_RUN_SUMMARY_PATH.read_text(encoding="utf-8"))
if training_summary.get("status") != "complete":
    raise ValueError("U10 학습 run_summary가 complete가 아닙니다.")
weights_sha256 = sha256(U10_WEIGHTS_PATH.read_bytes()).hexdigest()
if weights_sha256 != training_summary.get("weights_sha256"):
    raise ValueError("U10 best.pt와 training run_summary의 weights_sha256가 다릅니다.")
if int(training_summary["image_size"]) != U10_EVALUATION_IMAGE_SIZE:
    raise ValueError("학습과 평가 image size가 다릅니다.")

device_info = resolve_inference_device(U10_REQUESTED_DEVICE)
actual_device = device_info["actual_device"]
u10_inference_rows = []
if RUN_U10_EVALUATION_INFERENCE:
    verify_ultralytics_safe_load()
    import ultralytics

    evaluation_model = ultralytics.YOLO(str(U10_WEIGHTS_PATH))
    for row in pilot_rows.itertuples(index=False):
        video_id = str(row.video_id)
        output_directory = U10_EVALUATION_ROOT / video_id
        output_directory.mkdir(parents=True, exist_ok=True)
        for stale_name in ("plate_candidates.csv", "plate_track.csv", "run_summary.json"):
            (output_directory / stale_name).unlink(missing_ok=True)
        started = time.perf_counter()
        candidates, processed_frame_count = infer_plate_video_candidates(
            evaluation_model,
            video_path=PROJECT_ROOT / str(row.path),
            video_id=video_id,
            frame_count=int(row.frame_count),
            fps=float(row.fps),
            image_width=int(row.width),
            image_height=int(row.height),
            image_size=U10_EVALUATION_IMAGE_SIZE,
            confidence=U10_EVALUATION_CONFIDENCE,
            device=actual_device,
        )
        save_plate_candidate_cache(candidates, output_directory / "plate_candidates.csv")
        run_summary = {
            "video_id": video_id,
            "group": str(row.group),
            "device": device_info,
            "plate": {
                "status": "complete",
                "track_status": "awaiting_confirmation",
                "model": str(U10_WEIGHTS_PATH.relative_to(PROJECT_ROOT)),
                "weights_sha256": weights_sha256,
                "image_size": U10_EVALUATION_IMAGE_SIZE,
                "confidence": U10_EVALUATION_CONFIDENCE,
                "processed_frame_count": processed_frame_count,
                "candidate_row_count": len(candidates),
                "elapsed_seconds": time.perf_counter() - started,
            },
        }
        save_json_atomically(run_summary, output_directory / "run_summary.json")
        u10_inference_rows.append({
            "video_id": video_id,
            "candidate_rows": len(candidates),
            "status": "complete",
        })
else:
    for row in pilot_rows.itertuples(index=False):
        summary_path = U10_EVALUATION_ROOT / str(row.video_id) / "run_summary.json"
        if summary_path.exists():
            stored = json.loads(summary_path.read_text(encoding="utf-8"))
            u10_inference_rows.append({
                "video_id": str(row.video_id),
                "candidate_rows": stored.get("plate", {}).get("candidate_row_count", 0),
                "status": stored.get("plate", {}).get("status", "not_run"),
            })
        else:
            u10_inference_rows.append({
                "video_id": str(row.video_id),
                "candidate_rows": 0,
                "status": "not_run",
            })
display(pd.DataFrame(u10_inference_rows))

u10_frame_evaluation = pd.DataFrame()
plate_model_comparison = pd.DataFrame()
if CONFIRM_U10_PRIMARY_PLATES:
    expected_video_ids = set(pilot_rows["video_id"].astype(str))
    if set(U10_PRIMARY_PLATE_CHOICES) != expected_video_ids:
        raise ValueError("A/B/C 세 영상의 U10_PRIMARY_PLATE_CHOICES를 모두 입력하세요.")
    baseline_comparison = pd.read_csv(U9_COMPARISON_PATH)
    baseline_rows = baseline_comparison.loc[
        baseline_comparison["experiment_id"].eq("conf010")
    ].copy()
    frame_tables = []
    summary_rows = []
    for row in pilot_rows.itertuples(index=False):
        video_id = str(row.video_id)
        output_directory = U10_EVALUATION_ROOT / video_id
        summary_path = output_directory / "run_summary.json"
        run_summary = json.loads(summary_path.read_text(encoding="utf-8"))
        plate_summary = run_summary["plate"]
        baseline_video = baseline_rows.loc[baseline_rows["video_id"].eq(video_id)].iloc[0]
        fixed_conditions = {
            "image_size": int(plate_summary["image_size"]) == int(baseline_video["image_size"]),
            "confidence": abs(float(plate_summary["confidence"]) - float(baseline_video["confidence_threshold"])) <= 1e-9,
            "device": str(run_summary["device"]["actual_device"]) == str(baseline_video["device"]),
            "processed_frame_count": int(plate_summary["processed_frame_count"]) == int(row.frame_count),
        }
        if not all(fixed_conditions.values()):
            raise ValueError(f"{video_id}: U9와 U10의 고정 평가 조건이 다릅니다: {fixed_conditions}")
        if str(plate_summary["weights_sha256"]) == str(baseline_video["weights_sha256"]):
            raise ValueError(f"{video_id}: U9와 U10 weight가 같아 비교할 수 없습니다.")
        candidates = pd.read_csv(output_directory / "plate_candidates.csv")
        choice = U10_PRIMARY_PLATE_CHOICES[video_id]
        track_path = output_directory / "plate_track.csv"
        for stale_key in (
            "track_reason", "initial_frame_index", "initial_candidate_id", "tracked_frame_count"
        ):
            plate_summary.pop(stale_key, None)
        if choice is None:
            track_path.unlink(missing_ok=True)
            plate_track = None
            plate_summary.update({
                "track_status": "not_available",
                "track_reason": "primary_plate_not_detected",
            })
        else:
            pose_keypoints = pd.read_csv(U9_BASELINE_ROOT / video_id / "pose_keypoints.csv")
            lifter_centers = (
                pose_keypoints[["frame_index", "bbox_center_x", "bbox_center_y"]]
                .dropna()
                .drop_duplicates("frame_index")
                .rename(columns={"bbox_center_x": "center_x_px", "bbox_center_y": "center_y_px"})
            )
            initial_frame_index, initial_candidate_id = map(int, choice)
            plate_track = select_primary_plate_track(
                candidates,
                frame_count=int(row.frame_count),
                initial_frame_index=initial_frame_index,
                initial_candidate_id=initial_candidate_id,
                lifter_centers=lifter_centers,
                fps=float(row.fps),
            )
            save_plate_track(plate_track, track_path)
            plate_summary.update({
                "track_status": "complete",
                "initial_frame_index": initial_frame_index,
                "initial_candidate_id": initial_candidate_id,
                "tracked_frame_count": int(plate_track["status"].eq("ok").sum()),
            })
        video_ground_truth = u10_ground_truth.loc[
            u10_ground_truth["video_id"].astype(str).eq(video_id)
        ]
        frame_evaluation, video_summary = evaluate_plate_video(
            video_ground_truth,
            candidates,
            plate_track,
            frame_count=int(row.frame_count),
            fps=float(row.fps),
            iou_threshold=PLATE_EVALUATION_IOU_THRESHOLD,
            missing_track_reason=str(plate_summary.get("track_reason") or "no_confirmed_track"),
        )
        frame_evaluation.insert(0, "model_stage", "u10_candidate")
        frame_evaluation.insert(1, "group", str(row.group))
        frame_tables.append(frame_evaluation)
        video_summary.update({
            "model_stage": "u10_candidate",
            "group": str(row.group),
            "video_count": 1,
            "model": str(plate_summary["model"]),
            "weights_sha256": str(plate_summary["weights_sha256"]),
            "image_size": int(plate_summary["image_size"]),
            "confidence_threshold": float(plate_summary["confidence"]),
            "device": str(run_summary["device"]["actual_device"]),
            "candidate_row_count": len(candidates),
            "inference_seconds": float(plate_summary["elapsed_seconds"]),
        })
        summary_rows.append(video_summary)
        save_json_atomically(run_summary, summary_path)

    u10_per_video = pd.DataFrame(summary_rows)
    u10_overall = aggregate_plate_evaluations(u10_per_video)
    u10_overall.update({
        "model_stage": "u10_candidate",
        "group": "ALL",
        "candidate_row_count": int(u10_per_video["candidate_row_count"].sum()),
        "model": u10_per_video["model"].drop_duplicates().item(),
        "weights_sha256": u10_per_video["weights_sha256"].drop_duplicates().item(),
        "image_size": int(u10_per_video["image_size"].drop_duplicates().item()),
        "confidence_threshold": float(u10_per_video["confidence_threshold"].drop_duplicates().item()),
        "device": u10_per_video["device"].drop_duplicates().item(),
        "inference_seconds": float(u10_per_video["inference_seconds"].sum()),
    })
    u10_summary = pd.concat([u10_per_video, pd.DataFrame([u10_overall])], ignore_index=True)
    baseline_rows["model_stage"] = "u9_baseline"
    plate_model_comparison = pd.concat([baseline_rows, u10_summary], ignore_index=True, sort=False)
    u10_frame_evaluation = pd.concat(frame_tables, ignore_index=True)
    U10_EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
    u10_frame_evaluation.to_csv(U10_FRAME_EVALUATION_PATH, index=False)
    plate_model_comparison.to_csv(U10_MODEL_COMPARISON_PATH, index=False)
    display(plate_model_comparison[[
        "model_stage", "group", "video_id", "candidate_correct_count",
        "track_correct_count", "wrong_target_count", "plate_ok_rate",
        "longest_non_ok_gap_frames", "mean_normalized_center_error",
    ]].sort_values(["model_stage", "group"]))
else:
    print("초기 후보 확인 후 U10_PRIMARY_PLATE_CHOICES를 입력하고 CONFIRM_U10_PRIMARY_PLATES=True로 바꾸세요.")

## 14. 초기 원판 후보 확인

각 영상에서 처음 후보가 나온 frame의 모든 bbox와 candidate ID를 확인합니다. 올바른 주동작 원판의 (frame_index, candidate_id)를 설정 셀에 입력하며, 올바른 후보가 전혀 없으면 None을 입력합니다.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(18, 7))
for axis, row in zip(axes, pilot_rows.itertuples(index=False), strict=True):
    video_id = str(row.video_id)
    candidate_path = U10_EVALUATION_ROOT / video_id / "plate_candidates.csv"
    axis.axis("off")
    if not candidate_path.exists():
        axis.set_title(f"{video_id}: 후보 미생성")
        continue
    candidates = pd.read_csv(candidate_path)
    if candidates.empty:
        axis.set_title(f"{video_id}: 원판 후보 없음 → None")
        continue
    first_frame_index = int(candidates["frame_index"].min())
    first_candidates = candidates.loc[candidates["frame_index"].eq(first_frame_index)]
    frame = read_video_frame(PROJECT_ROOT / str(row.path), first_frame_index).copy()
    for candidate in first_candidates.itertuples(index=False):
        start = (int(candidate.bbox_x1), int(candidate.bbox_y1))
        end = (int(candidate.bbox_x2), int(candidate.bbox_y2))
        cv2.rectangle(frame, start, end, (0, 255, 0), 3)
        cv2.putText(
            frame,
            f"id={int(candidate.candidate_id)} conf={candidate.bbox_confidence:.2f}",
            start,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2,
        )
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(f"{video_id} | frame {first_frame_index}")
figure.tight_layout()
U10_EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
u10_initial_review_path = U10_EVALUATION_ROOT / "initial_plate_candidate_review.png"
figure.savefig(u10_initial_review_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"초기 후보 확인 그림: {u10_initial_review_path}")

## 15. count 기반 채택 판단

주 지표는 candidate correct count입니다. 12/30에서 최소 15/30으로 늘고, wrong target 0·track 비회귀·그룹별 회귀 제한·coverage 또는 gap 개선을 모두 통과할 때만 adopt합니다.

In [ ]:
u10_decision = {}
if plate_model_comparison.empty and U10_MODEL_COMPARISON_PATH.exists():
    plate_model_comparison = pd.read_csv(U10_MODEL_COMPARISON_PATH)
if plate_model_comparison.empty:
    print("결정 대기: 먼저 CONFIRM_U10_PRIMARY_PLATES=True로 평가를 실행하세요.")
else:
    baseline_overall = plate_model_comparison.loc[
        plate_model_comparison["model_stage"].eq("u9_baseline")
        & plate_model_comparison["group"].eq("ALL")
    ].iloc[0]
    candidate_overall = plate_model_comparison.loc[
        plate_model_comparison["model_stage"].eq("u10_candidate")
        & plate_model_comparison["group"].eq("ALL")
    ].iloc[0]
    baseline_groups = plate_model_comparison.loc[
        plate_model_comparison["model_stage"].eq("u9_baseline")
        & plate_model_comparison["group"].isin(["A", "B", "C"])
    ].set_index("group")
    candidate_groups = plate_model_comparison.loc[
        plate_model_comparison["model_stage"].eq("u10_candidate")
        & plate_model_comparison["group"].isin(["A", "B", "C"])
    ].set_index("group")

    baseline_candidate_correct_count = int(baseline_overall["candidate_correct_count"])
    candidate_candidate_correct_count = int(candidate_overall["candidate_correct_count"])
    baseline_track_correct_count = int(baseline_overall["track_correct_count"])
    candidate_track_correct_count = int(candidate_overall["track_correct_count"])
    candidate_wrong_target_count = int(candidate_overall["wrong_target_count"])
    baseline_group_counts = {
        group: int(baseline_groups.loc[group, "candidate_correct_count"])
        for group in ("A", "B", "C")
    }
    candidate_group_counts = {
        group: int(candidate_groups.loc[group, "candidate_correct_count"])
        for group in ("A", "B", "C")
    }
    minimum_correct_gain = 3
    correct_gain = candidate_candidate_correct_count - baseline_candidate_correct_count
    no_large_group_regression = all(
        candidate_group_counts[group] >= baseline_group_counts[group] - 1
        for group in ("A", "B", "C")
    )
    track_not_regressed = candidate_track_correct_count >= baseline_track_correct_count
    coverage_or_gap_improved = (
        float(candidate_overall["plate_ok_rate"]) > float(baseline_overall["plate_ok_rate"])
        or int(candidate_overall["longest_non_ok_gap_frames"]) < int(baseline_overall["longest_non_ok_gap_frames"])
    )

    if (
        correct_gain >= minimum_correct_gain
        and candidate_wrong_target_count == 0
        and no_large_group_regression
        and track_not_regressed
        and coverage_or_gap_improved
    ):
        decision = "adopt"
    elif correct_gain in {1, 2} or not no_large_group_regression:
        decision = "hold"
    else:
        decision = "reject"

    u10_decision = {
        "decision": decision,
        "minimum_correct_gain": minimum_correct_gain,
        "baseline_candidate_correct_count": baseline_candidate_correct_count,
        "candidate_candidate_correct_count": candidate_candidate_correct_count,
        "correct_gain": correct_gain,
        "baseline_track_correct_count": baseline_track_correct_count,
        "candidate_track_correct_count": candidate_track_correct_count,
        "candidate_wrong_target_count": candidate_wrong_target_count,
        "baseline_group_counts": baseline_group_counts,
        "candidate_group_counts": candidate_group_counts,
        "no_large_group_regression": no_large_group_regression,
        "track_not_regressed": track_not_regressed,
        "coverage_or_gap_improved": coverage_or_gap_improved,
        "baseline_plate_ok_rate": float(baseline_overall["plate_ok_rate"]),
        "candidate_plate_ok_rate": float(candidate_overall["plate_ok_rate"]),
        "baseline_longest_gap": int(baseline_overall["longest_non_ok_gap_frames"]),
        "candidate_longest_gap": int(candidate_overall["longest_non_ok_gap_frames"]),
    }
    U10_EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
    save_json_atomically(u10_decision, U10_DECISION_PATH)
    display(pd.DataFrame([u10_decision]))

## 16. 실패 frame contact sheet

수동 정답은 초록색, 선택 track은 빨간색으로 표시합니다. candidate 또는 track이 맞지 않은 frame만 모아 수치가 실제 실패 모습을 반영하는지 검수합니다.

In [ ]:
u10_failure_contact_sheet_paths = []
if u10_frame_evaluation.empty and U10_FRAME_EVALUATION_PATH.exists():
    u10_frame_evaluation = pd.read_csv(U10_FRAME_EVALUATION_PATH)
if u10_frame_evaluation.empty:
    print("실패 검수 대기: 먼저 A/B/C 평가를 완료하세요.")
else:
    failures = u10_frame_evaluation.loc[
        u10_frame_evaluation["plate_present"].eq("yes")
        & (
            u10_frame_evaluation["candidate_outcome"].ne("correct")
            | u10_frame_evaluation["track_outcome"].ne("correct")
        )
    ].copy()
    failures["review_priority"] = failures["track_outcome"].map({
        "wrong_target": 0,
        "missing": 1,
        "correct": 2,
    })
    failures = failures.sort_values(
        ["review_priority", "track_iou", "video_id", "frame_index"],
        na_position="last",
    ).head(U10_FAILURE_REVIEW_LIMIT)
    if failures.empty:
        print("검토할 U10 실패 frame이 없습니다.")
    else:
        metadata_by_video = pilot_rows.set_index("video_id")
        page_size = 10
        column_count = 5
        for page_index, start in enumerate(range(0, len(failures), page_size), start=1):
            page_failures = failures.iloc[start:start + page_size]
            review_frames = {}
            for video_id, video_failures in page_failures.groupby("video_id"):
                metadata = metadata_by_video.loc[video_id]
                target_indices = set(video_failures["frame_index"].astype(int))
                for frame_index, _, frame in iter_video_frames(
                    PROJECT_ROOT / str(metadata["path"]),
                    expected_frame_count=int(metadata["frame_count"]),
                ):
                    if frame_index in target_indices:
                        review_frames[(video_id, frame_index)] = frame.copy()

            row_count = math.ceil(len(page_failures) / column_count)
            figure, axes = plt.subplots(
                row_count, column_count, figsize=(20, 4 * row_count), squeeze=False
            )
            for axis in axes.flat:
                axis.axis("off")
            for axis, failure in zip(axes.flat, page_failures.itertuples(index=False)):
                frame = review_frames[(failure.video_id, int(failure.frame_index))]
                ground_truth_start = (
                    int(failure.ground_truth_bbox_x1), int(failure.ground_truth_bbox_y1)
                )
                ground_truth_end = (
                    int(failure.ground_truth_bbox_x2), int(failure.ground_truth_bbox_y2)
                )
                cv2.rectangle(frame, ground_truth_start, ground_truth_end, (0, 255, 0), 4)
                if failure.track_status == "ok":
                    track_start = (int(failure.track_bbox_x1), int(failure.track_bbox_y1))
                    track_end = (int(failure.track_bbox_x2), int(failure.track_bbox_y2))
                    cv2.rectangle(frame, track_start, track_end, (0, 0, 255), 4)
                axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                axis.set_title(
                    f"{failure.video_id} frame={int(failure.frame_index)} "
                    f"candidate={failure.candidate_outcome}, track={failure.track_outcome}"
                )
            figure.suptitle("Green: ground truth | Red: selected track")
            figure.tight_layout()
            filename = (
                "failure_contact_sheet.png"
                if len(failures) <= page_size
                else f"failure_contact_sheet_{page_index:02d}.png"
            )
            output_path = U10_EVALUATION_ROOT / filename
            figure.savefig(output_path, dpi=140, bbox_inches="tight")
            u10_failure_contact_sheet_paths.append(output_path)
            plt.show()
            plt.close(figure)
        display(pd.DataFrame({"failure_contact_sheet_path": u10_failure_contact_sheet_paths}))

## 다음 checkpoint

후보 추론 → 초기 후보 확인 → 세 선택값 입력 → 평가 → 결정 → 실패 contact sheet 순서로 실행합니다. 실제 결과는 `decision.json`과 `plate_model_comparison.csv`에 남습니다.